# Moments: the candidates, registered and written for submission

Settings + plane → the 13 moments (`models/moments_film_mlp.py`). Five candidates, each trained on
the split of `moments_task/01_moments.ipynb` (injected configs, seed 12: 60% train, 20% validation
for the epochs, 20% test), scored on the test configs and registered; then retrained on all injected
configs, registered with `_all`, loaded back from the registry and written as a submission:

| candidate | what |
|---|---|
| `mlp` | the MLP of `01_moments.ipynb` (the file handed in so far), retrained here |
| `mlp_physics` | the same + 4 physics inputs: n_plateau, z_from_end, n_e, dE_gain |
| `film_centroid_term` | FiLM Gaussian, loss + 1 × MSE of the four transverse centroids |
| `hybrid_film_mlp_physics` | FiLM Gaussian without the term; the four centroids from `mlp_physics` |
| `hybrid_film_avg_centroids` | the same, centroids averaged over `mlp_physics` and `film_centroid_term` |

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import copy
import shutil
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch

import pallas_pipeline as pp
import pallas_score

OWNER = "m"
CASE = "moments"
MODEL_FILE = "models/moments_film_mlp.py"
FV_NAME, FV_VERSION = "m_moments_raw_fv", 1
SEED = 12
DEVICE = "cuda:1"
EPOCHS = 400
REGISTER = False

torch.set_num_threads(16)
project = pp.connect(pp.PROJECT)
fs = project.get_feature_store()
module = pp.load_model_file(MODEL_FILE)
TARGETS = module.targets
started = time.time()

/home/mlenivenko/miniconda3/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-10-15 14:19:21,513 INFO: Initializing external client


2026-10-15 14:19:21,514 INFO: Base URL: https://aissai26.hops.live:443


2026-10-15 14:19:22,714 INFO: Python Engine initialized.



Logged in to project, explore it here https://aissai26.hops.live:443/p/2170


## 1. Rows and split

A feature view over `pallas_b_moments`, `particle_group == "all"`: config, plane, settings, charge,
the six means and the 21 covariance entries (FiLM needs the off-diagonal ones). Rows without a bunch
(`q_tot_pC = 0`) and non-injected configs are dropped; the widths are √ of the covariance diagonal.

In [2]:
COLUMNS = (["config", "plane_mm", "q_tot_pC"] + module.SETTINGS + [f"mean_{c}_um" for c in ["zeta", "y", "z"]]
           + ["mean_ux", "mean_uy", "mean_uz"] + [f"cov_{i}{j}" for i in range(6) for j in range(i, 6)])
fv = pp.get_feature_view(fs, FV_NAME, FV_VERSION)
if fv is None:
    fg = fs.get_feature_group("pallas_b_moments", version=1)
    query = fg.select([pp.fs_name(c) for c in COLUMNS]).filter(fg.particle_group == "all")
    fv = fs.create_feature_view(name=FV_NAME, version=FV_VERSION, query=query, training_helper_columns=["config"],
                                description="moments, group all: settings + plane_mm, charge, means, full covariance")
    print(f"created {FV_NAME} v{FV_VERSION}")
TD_VERSION = pp.prepare_statistics(fv)

rows = fv.query.read().rename(columns={pp.fs_name(c): c for c in COLUMNS})[COLUMNS]
rows = rows.rename(columns={f"mean_{c}_um": f"mean_{c}" for c in ["zeta", "y", "z"]})
for i, c in enumerate(module.COORDS):
    rows[f"sigma_{c}"] = np.sqrt(rows[f"cov_{i}{i}"])
b = fs.get_feature_group("pallas_campaign_b", version=1).select(["config", "injected"]).read()
rows = rows[(rows.q_tot_pC > 0) & rows.config.isin(b.config[b.injected])]
rows = rows.sort_values(["config", "plane_mm"]).reset_index(drop=True)

configs = np.sort(rows.config.unique())
order = np.random.default_rng(SEED).permutation(configs)
n_train, n_val = int(0.6 * len(configs)), int(0.2 * len(configs))
fold = {c: "train" for c in order[:n_train]} | {c: "val" for c in order[n_train:n_train + n_val]} | {c: "test" for c in order[n_train + n_val:]}
rows["fold"] = rows.config.map(fold)
train, val, test = (rows[rows.fold == f].reset_index(drop=True) for f in ["train", "val", "test"])
test_configs = [int(c) for c in order[n_train + n_val:]]
print(train.config.nunique(), "train,", val.config.nunique(), "validation,", test.config.nunique(), "test configs;",
      len(train), "/", len(val), "/", len(test), "rows")


def score(model, df):
    sub = pd.concat([df[["config", "plane_mm"]], model.predict(df)], axis=1)
    return pallas_score.score_submission(sub, df[["config", "plane_mm", "L_inj"] + TARGETS])

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.74s) 


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (0.49s) 
1729 train, 576 validation, 577 test configs; 34563 / 11515 / 11533 rows


## 2. Train on the split, score the test configs

About 40 minutes on one GPU, most of it the two FiLM ensembles.

In [3]:
parts = {"mlp": module.build_model("mlp", epochs=EPOCHS, device=DEVICE),
         "mlp_physics": module.build_model("mlp", use_physics=True, epochs=EPOCHS, device=DEVICE),
         "film": module.build_model("film", lambda_c=0.0, weighted=True, epochs=EPOCHS, device=DEVICE),
         "film_centroid_term": module.build_model("film", lambda_c=1.0, weighted=False, epochs=EPOCHS, device=DEVICE)}
for name, m in parts.items():
    print(name, flush=True)
    m.fit(train, val)
    print(f"--- {(time.time() - started) / 60:.1f} min", flush=True)


def candidates(parts):
    return {"mlp": parts["mlp"], "mlp_physics": parts["mlp_physics"], "film_centroid_term": parts["film_centroid_term"],
            "hybrid_film_mlp_physics": module.Hybrid(parts["film"], module.centroids, [parts["mlp_physics"]]),
            "hybrid_film_avg_centroids": module.Hybrid(parts["film"], module.centroids,
                                                       [parts["mlp_physics"], parts["film_centroid_term"]])}


models = candidates(parts)
table = pd.DataFrame({name: score(m, test) for name, m in models.items()}).T
table[TARGETS + ["mean"]].round(3)

mlp


  kept epoch 57, validation loss 0.1909


  kept epoch 54, validation loss 0.1910


  kept epoch 54, validation loss 0.1886


  kept epoch 57, validation loss 0.1887


...

  kept epoch 63, validation loss 1.4814


  kept epoch 69, validation loss 1.4705


  kept epoch 69, validation loss 1.4781


--- 39.2 min


,mean_zeta,mean_y,mean_z,mean_ux,mean_uy,mean_uz,sigma_zeta,sigma_y,sigma_z,sigma_ux,sigma_uy,sigma_uz,q_tot_pC,mean
mlp,0.954,0.433,0.176,0.995,0.724,0.496,0.956,0.977,0.970,0.974,0.970,0.948,0.995,0.813
mlp_physics,0.964,0.441,0.190,0.996,0.746,0.523,0.949,0.976,0.966,0.966,0.963,0.947,0.997,0.817
film_centroid_term,0.955,0.430,0.149,0.997,0.756,0.521,0.939,0.983,0.976,0.975,0.975,0.946,0.998,0.815
hybrid_film_mlp_physics,0.985,0.441,0.190,0.999,0.746,0.523,0.968,0.985,0.978,0.987,0.978,0.951,1.000,0.825
hybrid_film_avg_centroids,0.985,0.444,0.180,0.999,0.758,0.530,0.968,0.985,0.978,0.987,0.978,0.951,1.000,0.826


## 3. Register the split models

Metrics `holdout_*` are on the 577 test configs (the validation configs only picked epochs).
Each bundle also carries `physics_formulas.py`, for the physics inputs.

In [4]:
DESCRIPTIONS = {
    "mlp": "MLP 5 nets, settings + plane (01_moments.ipynb)",
    "mlp_physics": "MLP 5 nets, settings + plane + n_plateau, z_from_end, n_e, dE_gain",
    "film_centroid_term": "FiLM Gaussian 5 nets, NLL + log-q MSE + 1 x transverse-centroid MSE",
    "hybrid_film_mlp_physics": "FiLM Gaussian 5 nets (NLL + log-q MSE) with the 4 transverse centroids from mlp_physics",
    "hybrid_film_avg_centroids": "FiLM Gaussian 5 nets with the 4 transverse centroids averaged over mlp_physics and film_centroid_term"}


def register(model, name, meta, metrics, description):
    bundle = pp.save_bundle(model, MODEL_FILE, meta, Path("team_models") / name)
    shutil.copy("physics_formulas.py", bundle / "physics_formulas.py")
    if not REGISTER:
        return None
    entry = pp.register_model(project, bundle, name, fv, TD_VERSION, metrics, description)
    print(f"registered {name} v{entry.version}", flush=True)
    return entry


metrics, metas = {}, {}
for name, m in models.items():
    metrics[name] = {"holdout_mean_r2": table.loc[name, "mean"],
                     **{f"holdout_r2_{pp.fs_name(t)}": table.loc[name, t] for t in TARGETS}}
    metas[name] = {"case": CASE, "inputs": module.SETTINGS + ["plane_mm"], "targets": TARGETS,
                   "feature_view": FV_NAME, "feature_view_version": FV_VERSION, "td_version": TD_VERSION,
                   "model_file": Path(MODEL_FILE).stem, "candidate": name, "split": "random 60/20/20 by config",
                   "seed": SEED, "test_configs": test_configs,
                   "epochs": {k: list(p.epochs) for k, p in parts.items()},
                   "holdout_scores": {k: float(v) for k, v in metrics[name].items()}}
    register(m, f"{OWNER}_{CASE}_{name}", metas[name], metrics[name],
             f"{CASE}: {DESCRIPTIONS[name]}; {FV_NAME} v{FV_VERSION}, test = 20% of injected configs (seed {SEED})")

  0%|          | 0/6 [00:00<?, ?it/s]

Creating model folder:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model folder:  17%|█▋        | 1/6 [00:00<00:03,  1.33it/s]

Uploading model files:  17%|█▋        | 1/6 [00:00<00:03,  1.33it/s]

Uploading model files (0 dirs, 0 files):  17%|█▋        | 1/6 [00:00<00:03,  1.33it/s]

...

Waiting for model registration:  67%|██████▋   | 4/6 [00:08<00:03,  1.80s/it]

Waiting for model registration:  83%|████████▎ | 5/6 [00:14<00:03,  3.17s/it]

Model export complete:  83%|████████▎ | 5/6 [00:14<00:03,  3.17s/it]         

Model export complete: 100%|██████████| 6/6 [00:14<00:00,  2.36s/it]

Model created, explore it at https://aissai26.hops.live:443/p/2170/models/m_moments_hybrid_film_avg_centroids/1
registered m_moments_hybrid_film_avg_centroids v1


## 4. Retrain on all injected configs, register

Same epochs as step 2, no validation, scalers kept. The metrics attached are step 2's.

In [5]:
parts_all = copy.deepcopy(parts)
for name, m in parts_all.items():
    m.refit(rows)
print(f"--- retrained on {rows.config.nunique()} configs, {(time.time() - started) / 60:.1f} min")
models_all = candidates(parts_all)
registered = {}
for name, m in models_all.items():
    NAME_ALL = f"{OWNER}_{CASE}_{name}_all"
    meta = {**metas[name], "split": "all", "trained_on": "every injected configuration of the view",
            "holdout_scores_from": f"{OWNER}_{CASE}_{name} (the same recipe on the split)"}
    registered[name] = register(m, NAME_ALL, meta, metrics[name],
                                f"{CASE}: {DESCRIPTIONS[name]}; retrained on all injected configs of {FV_NAME} "
                                f"v{FV_VERSION} with the epochs of {OWNER}_{CASE}_{name}, whose test metrics these are")

--- retrained on 2882 configs, 61.3 min


  0%|          | 0/6 [00:00<?, ?it/s]

Creating model folder:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model folder:  17%|█▋        | 1/6 [00:00<00:04,  1.09it/s]

Uploading model files:  17%|█▋        | 1/6 [00:00<00:04,  1.09it/s]

...

Waiting for model registration:  67%|██████▋   | 4/6 [00:08<00:03,  1.78s/it]

Waiting for model registration:  83%|████████▎ | 5/6 [00:13<00:03,  3.07s/it]

Model export complete:  83%|████████▎ | 5/6 [00:13<00:03,  3.07s/it]         

Model export complete: 100%|██████████| 6/6 [00:13<00:00,  2.31s/it]

Model created, explore it at https://aissai26.hops.live:443/p/2170/models/m_moments_hybrid_film_avg_centroids_all/1
registered m_moments_hybrid_film_avg_centroids_all v1


## 5. Load each back from the registry, write the submissions

One folder per model under `submissions/moments/`, each with `submission_moments.csv` (3,213 rows of
`test_moments_planes.parquet`).

In [6]:
to_submit = pd.read_parquet(pp.DATA_DIR / "test_moments_planes.parquet")
case = SimpleNamespace(targets=TARGETS, ids=["config", "plane_mm"])
listing = {}
for name, m in models_all.items():
    NAME_ALL = f"{OWNER}_{CASE}_{name}_all"
    if registered[name] is not None:
        version = registered[name].version
        loaded, _ = pp.load_registered_model(project, NAME_ALL, version, "team_models")
    else:
        version, loaded = 0, m
    pred = loaded.predict(to_submit)
    assert np.allclose(pred.to_numpy(), m.predict(to_submit).to_numpy())        # registry copy = this one
    submission = pd.concat([to_submit[["config", "plane_mm"]].reset_index(drop=True), pred], axis=1)
    path = pp.write_submission(case, submission, to_submit, Path("submissions") / CASE / f"{NAME_ALL}_v{version}" / "submission_moments.csv")
    listing[name] = {"test R2": round(table.loc[name, "mean"], 4), "model": f"{NAME_ALL} v{version}", "file": str(path)}
print(f"--- done {(time.time() - started) / 60:.1f} min")
pd.DataFrame(listing).T

Downloading: 0.000%|          | 0/122 elapsed<00:00 remaining<?

Downloading: 100.000%|██████████| 122/122 elapsed<00:00 remaining<00:00

Downloading: 0.000%|          | 0/20393 elapsed<00:00 remaining<?

Downloading: 100.000%|██████████| 20393/20393 elapsed<00:00 remaining<00:00

Downloading: 0.000%|          | 0/27702 elapsed<00:00 remaining<?

...

Downloading: 97.119%|█████████▋| 12569386/12942305 elapsed<00:05 remaining<00:00

Downloading: 99.237%|█████████▉| 12843578/12942305 elapsed<00:05 remaining<00:00

Downloading: 100.000%|██████████| 12942305/12942305 elapsed<00:05 remaining<00:00

--- done 63.4 min


,test R2,model,file
mlp,0.8129,m_moments_mlp_all v1,submissions/moments/m_moments_mlp_all_v1/submi...
mlp_physics,0.8172,m_moments_mlp_physics_all v1,submissions/moments/m_moments_mlp_physics_all_...
film_centroid_term,0.8153,m_moments_film_centroid_term_all v1,submissions/moments/m_moments_film_centroid_te...
hybrid_film_mlp_physics,0.8254,m_moments_hybrid_film_mlp_physics_all v1,submissions/moments/m_moments_hybrid_film_mlp_...
hybrid_film_avg_centroids,0.8263,m_moments_hybrid_film_avg_centroids_all v1,submissions/moments/m_moments_hybrid_film_avg_...
